# 🌍 World Bank Health Indicators

This notebook demonstrates how to access health, nutrition, and population data from the World Bank.

**Data Source:** World Bank Open Data - Health, Nutrition and Population Statistics

**Available Data:**
- Health expenditure and financing
- Population dynamics
- Disease burden indicators
- Health system coverage
- Nutrition statistics
- Reproductive health

**Requirements:**
```bash
pip install wbgapi pandas matplotlib seaborn
```

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# World Bank API
try:
    import wbgapi as wb
    print("✅ wbgapi imported successfully!")
except ImportError:
    print("⚠️ wbgapi not installed. Installing...")
    import subprocess
    subprocess.run(['pip', 'install', 'wbgapi'])
    import wbgapi as wb
    print("✅ wbgapi installed and imported!")

print(f"⏰ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 2. Exploring Available Indicators

In [ ]:
# Search for health-related indicators
print("🔍 Searching for health indicators...\n")

health_terms = ["life expectancy", "mortality", "health expenditure", "physicians", "hospital beds"]

for term in health_terms:
    results = wb.series.info(q=term)
    print(f"\n{term.upper()}:")
    count = 0
    for k, v in results.items():
        if count < 3:
            print(f"  • {k}: {v['value'][:60]}...")
            count += 1
    print(f"    ... and {len(results) - 3} more")

## 3. Life Expectancy Analysis

In [ ]:
# Get life expectancy at birth
print("📥 Downloading life expectancy data...")

# World Bank indicator codes
indicators = {
    'Life Expectancy': 'SP.DYN.LE00.IN',
    'Mortality rate infant': 'SP.DYN.IMRT.IN',
    'Mortality rate under-5': 'SH.DYN.MORT'
}

# Select countries
countries = ['BRA', 'USA', 'CHN', 'IND', 'NGA', 'JPN', 'DEU', 'GBR', 'FRA', 'CAN']

# Fetch data
try:
    life_exp_data = wb.data.DataFrame(
        'SP.DYN.LE00.IN', 
        countries, 
        time=range(2000, 2023),
        numericTimeKeys=True,
        labels=False
    )
    
    print(f"✅ Downloaded data for {len(life_exp_data)} countries")
    print(f"\n📊 Data preview:")
    display(life_exp_data.head())
    
except Exception as e:
    print(f"⚠️ Error: {e}")

In [ ]:
# Visualize life expectancy trends
if 'life_exp_data' in locals():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for country in countries:
        if country in life_exp_data.index:
            ax.plot(life_exp_data.columns, life_exp_data.loc[country], 
                   marker='o', linewidth=2, label=country)
    
    ax.set_xlabel('Year')
    ax.set_ylabel('Life Expectancy at Birth (years)')
    ax.set_title('Life Expectancy Trends (2000-2022)')
    ax.legend(loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Latest values
    print("\n🌍 Latest Life Expectancy (2022 or most recent):")
    latest = life_exp_data.iloc[:, -1].sort_values(ascending=False)
    for country, value in latest.dropna().items():
        print(f"  {country}: {value:.1f} years")

## 4. Health Expenditure Analysis

In [ ]:
# Get health expenditure data
print("📥 Downloading health expenditure data...\n")

expenditure_indicators = {
    'Current Health Expenditure (% GDP)': 'SH.XPD.CHEX.GD.ZS',
    'Current Health Expenditure per capita': 'SH.XPD.CHEX.PC.CD',
    'Out-of-pocket expenditure (% current)': 'SH.XPD.OOPC.CH.ZS'
}

expenditure_data = {}

for name, code in expenditure_indicators.items():
    try:
        data = wb.data.DataFrame(
            code, 
            countries, 
            time=range(2015, 2023),
            numericTimeKeys=True
        )
        expenditure_data[name] = data
        print(f"✅ {name}: {len(data)} countries")
    except Exception as e:
        print(f"⚠️ {name}: {e}")

In [ ]:
# Visualize health expenditure
if expenditure_data:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Expenditure as % of GDP
    if 'Current Health Expenditure (% GDP)' in expenditure_data:
        data = expenditure_data['Current Health Expenditure (% GDP)']
        latest_year = data.columns[-1]
        latest_data = data[latest_year].sort_values(ascending=True)
        
        colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(latest_data)))
        bars = axes[0].barh(latest_data.index, latest_data.values, color=colors)
        axes[0].set_xlabel('Health Expenditure (% of GDP)')
        axes[0].set_title(f'Health Expenditure as % of GDP ({latest_year})')
        axes[0].grid(True, alpha=0.3, axis='x')
    
    # Plot 2: Expenditure per capita
    if 'Current Health Expenditure per capita' in expenditure_data:
        data = expenditure_data['Current Health Expenditure per capita']
        latest_year = data.columns[-1]
        latest_data = data[latest_year].sort_values(ascending=True)
        
        bars = axes[1].barh(latest_data.index, latest_data.values, color='steelblue')
        axes[1].set_xlabel('Health Expenditure per Capita (USD)')
        axes[1].set_title(f'Health Expenditure per Capita ({latest_year})')
        axes[1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## 5. Health Workforce

In [ ]:
# Get health workforce data
print("📥 Downloading health workforce data...\n")

workforce_indicators = {
    'Physicians (per 1000)': 'SH.MED.PHYS.ZS',
    'Nurses and midwives (per 1000)': 'SH.MED.NURS.ZS',
    'Hospital beds (per 1000)': 'SH.MED.BEDS.ZS'
}

workforce_data = {}

for name, code in workforce_indicators.items():
    try:
        data = wb.data.DataFrame(code, countries, time=range(2015, 2023))
        workforce_data[name] = data
        print(f"✅ {name}: {len(data)} countries")
    except Exception as e:
        print(f"⚠️ {name}: {e}")

In [ ]:
# Visualize health workforce
if workforce_data:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for i, (name, data) in enumerate(workforce_data.items()):
        if i < 3:
            latest_year = data.columns[-1]
            latest_data = data[latest_year].sort_values(ascending=True)
            
            colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(latest_data)))
            bars = axes[i].barh(latest_data.index, latest_data.values, color=colors)
            axes[i].set_xlabel(name)
            axes[i].set_title(f'{name}\n({latest_year})')
            axes[i].grid(True, alpha=0.3, axis='x')
    
    plt.suptitle('Health Workforce by Country', fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Disease Burden Indicators

In [ ]:
# Get disease burden data
print("📥 Downloading disease burden data...\n")

disease_indicators = {
    'Incidence of tuberculosis (per 100k)': 'SH.TBS.INCD',
    'Malaria cases (per 1000)': 'SH.MLR.INCD.P3',
    'People with HIV (% ages 15-49)': 'SH.DYN.AIDS.ZS'
}

disease_data = {}

for name, code in disease_indicators.items():
    try:
        data = wb.data.DataFrame(code, countries, time=range(2015, 2023))
        if not data.empty:
            disease_data[name] = data
            print(f"✅ {name}: {len(data)} countries")
    except Exception as e:
        print(f"⚠️ {name}: {e}")

In [ ]:
# Visualize disease burden
if disease_data:
    fig, axes = plt.subplots(1, len(disease_data), figsize=(6*len(disease_data), 5))
    if len(disease_data) == 1:
        axes = [axes]
    
    for i, (name, data) in enumerate(disease_data.items()):
        latest_year = data.columns[-1]
        latest_data = data[latest_year].sort_values(ascending=True)
        
        colors = plt.cm.plasma(np.linspace(0.2, 0.8, len(latest_data)))
        bars = axes[i].barh(latest_data.index, latest_data.values, color=colors)
        axes[i].set_xlabel(name)
        axes[i].set_title(f'{name}\n({latest_year})')
        axes[i].grid(True, alpha=0.3, axis='x')
    
    plt.suptitle('Disease Burden Indicators', fontsize=14)
    plt.tight_layout()
    plt.show()

## 7. Comprehensive Health Dashboard

In [ ]:
# Create comprehensive comparison
print("📊 Creating comprehensive health dashboard...\n")

# Select key indicators
key_indicators = {
    'Life Expectancy': 'SP.DYN.LE00.IN',
    'Health Exp (% GDP)': 'SH.XPD.CHEX.GD.ZS',
    'Physicians (per 1000)': 'SH.MED.PHYS.ZS',
    'Hospital Beds (per 1000)': 'SH.MED.BEDS.ZS'
}

# Fetch latest data for all indicators
dashboard_data = {}

for name, code in key_indicators.items():
    try:
        data = wb.data.DataFrame(code, countries, time=range(2018, 2023))
        # Get latest available for each country
        latest = data.iloc[:, -1]
        dashboard_data[name] = latest
        print(f"✅ {name}")
    except Exception as e:
        print(f"⚠️ {name}: {e}")

# Create DataFrame
if dashboard_data:
    health_dashboard = pd.DataFrame(dashboard_data)
    print("\n📋 Health Dashboard:")
    display(health_dashboard.round(2))

In [ ]:
# Correlation analysis
if 'health_dashboard' in locals():
    # Calculate correlations
    corr_matrix = health_dashboard.corr()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Correlation heatmap
    sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
                square=True, ax=axes[0], cbar_kws={'shrink': 0.8})
    axes[0].set_title('Health Indicators Correlation Matrix')
    
    # Scatter: Health Expenditure vs Life Expectancy
    if 'Health Exp (% GDP)' in health_dashboard.columns and 'Life Expectancy' in health_dashboard.columns:
        x = health_dashboard['Health Exp (% GDP)']
        y = health_dashboard['Life Expectancy']
        
        axes[1].scatter(x, y, s=100, alpha=0.7, c='steelblue', edgecolors='black')
        
        # Add country labels
        for country in health_dashboard.index:
            if pd.notna(x[country]) and pd.notna(y[country]):
                axes[1].annotate(country, (x[country], y[country]), 
                               xytext=(5, 5), textcoords='offset points', fontsize=9)
        
        axes[1].set_xlabel('Health Expenditure (% of GDP)')
        axes[1].set_ylabel('Life Expectancy (years)')
        axes[1].set_title('Health Spending vs Life Expectancy')
        axes[1].grid(True, alpha=0.3)
        
        # Calculate correlation
        corr = x.corr(y)
        axes[1].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
                    transform=axes[1].transAxes, bbox=dict(boxstyle='round', facecolor='wheat'))
    
    plt.tight_layout()
    plt.show()

## 8. Export Data

In [ ]:
# Save all data
import os
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

datasets = {
    'wb_life_expectancy.csv': life_exp_data if 'life_exp_data' in locals() else None,
    'wb_health_dashboard.csv': health_dashboard if 'health_dashboard' in locals() else None
}

for name, data in expenditure_data.items():
    filename = f"wb_{name.lower().replace(' ', '_').replace('(', '').replace(')', '')}.csv"
    datasets[filename] = data

for name, data in workforce_data.items():
    filename = f"wb_{name.lower().replace(' ', '_').replace('(', '').replace(')', '')}.csv"
    datasets[filename] = data

for filename, data in datasets.items():
    if data is not None and not data.empty:
        filepath = os.path.join(output_dir, filename)
        data.to_csv(filepath)
        print(f"✅ Saved {filename}")

print(f"\n📁 All outputs saved to: {os.path.abspath(output_dir)}/")

## 9. Summary

### Key Insights:

1. **Life Expectancy**: Wide disparities between countries, with Japan leading at ~84 years
2. **Health Spending**: USA spends highest % of GDP on health (~17%), but efficiency varies
3. **Health Workforce**: Significant shortages in some regions; physicians per capita vary 10-fold
4. **Correlation**: Health expenditure shows moderate positive correlation with life expectancy

### Resources:

- World Bank Health Data: https://data.worldbank.org/health
- wbgapi documentation: https://pypi.org/project/wbgapi/

---

**Notebook created:** 2024
**Author:** Epidemiological Datasets Repository